# BM25 on Cranfield with OKAPI-PACK

Full pipeline: index Cranfield → BM25 retrieval → NDCG evaluation.

> **Run this in GitHub Codespaces** (x86). The OKAPI binaries are 32-bit x86 and crash under QEMU on Apple Silicon.

See [CRANFIELD.md](CRANFIELD.md) for a detailed explanation of each step.

## 1. Install dependencies

In [ ]:
!pip install -q ir_datasets ir_measures

## 2. Configuration

In [ ]:
import os
import subprocess
import re
from pathlib import Path

# OKAPI install root (inside the container)
OKAPI_ROOT    = Path('/okapi-pack/okapi-pack')
BSS_PARMPATH  = str(OKAPI_ROOT / 'databases')

# Where we store Cranfield data files
DATA_DIR  = Path('/data/cranfield')
BIB_DIR   = DATA_DIR / 'bibfiles'
EXCH_FILE = DATA_DIR / 'cranfield.exch'

DB_NAME = 'cranfield'

DATA_DIR.mkdir(parents=True, exist_ok=True)
BIB_DIR.mkdir(parents=True, exist_ok=True)

# Make OKAPI binaries available in PATH
os.environ['BSS_PARMPATH'] = BSS_PARMPATH
os.environ['BSS_TEMPPATH'] = '/tmp'
os.environ['PATH'] = str(OKAPI_ROOT / 'bin') + ':' + os.environ.get('PATH', '')
os.environ['LD_LIBRARY_PATH'] = '/usr/lib/i386-linux-gnu/'

print('BSS_PARMPATH:', BSS_PARMPATH)
print('BIB_DIR:     ', BIB_DIR)
print('EXCH_FILE:   ', EXCH_FILE)

## 3. Write database descriptor files

OKAPI needs four config files in `$BSS_PARMPATH` for each database.

In [ ]:
db_dir = Path(BSS_PARMPATH)

# ── 3a. Main descriptor ───────────────────────────────────────────────────────
descriptor = f"""name={DB_NAME}
lastbibvol=0
bib_basename={DB_NAME}.bib
bib_dir={BIB_DIR}/
bibsize=2047
real_bibsize=0
display_name={DB_NAME}
explanation=Cranfield Collection 1400 documents
nr=0
nf=2
f_abbrev=DN
f_abbrev=TX
rec_mult=4
fixed=0
db_type=ai
has_lims=0
maxreclen=65536
ni=2
last_ixvol=0
ix_stem={BIB_DIR}/{DB_NAME}
ix_volsize=2047
ix_type=9
last_ixvol=0
ix_stem={BIB_DIR}/{DB_NAME}
ix_volsize=2047
ix_type=9
"""
(db_dir / DB_NAME).write_text(descriptor)

# ── 3b. Field types ───────────────────────────────────────────────────────────
(db_dir / f'{DB_NAME}.field_types').write_text(
    "1 LITERAL_NC\n2 TEXT\n"
)

# ── 3c. Search groups ─────────────────────────────────────────────────────────
(db_dir / f'{DB_NAME}.search_groups').write_text(
    "kw 1 0 words3 sstem gsl.empty 2 0 -1\n"
    "dn 1 1 literal nostem gsl.empty 1 0 -1\n"
)

# ── 3d. Grant access in db_avail ──────────────────────────────────────────────
db_avail = db_dir / 'db_avail'
content = db_avail.read_text() if db_avail.exists() else ''
if f'{DB_NAME} *' not in content:
    db_avail.open('a').write(f'{DB_NAME} *\n')

print('Descriptor files written to', BSS_PARMPATH)

## 4. Load Cranfield and write exchange format

Exchange format: `<docno>\x1e<title> <text>\x1d\n`

In [ ]:
import ir_datasets

FIELD_MARK  = '\x1e'  # 0x1E — separates fields within a record
RECORD_MARK = '\x1d'  # 0x1D — terminates each record

dataset = ir_datasets.load('cranfield')

with EXCH_FILE.open('w', encoding='utf-8') as f:
    for doc in dataset.docs_iter():
        # Combine title + text as a single searchable field
        content = f'{doc.title} {doc.text}'
        # Sanitise: remove any stray field/record markers from the text itself
        content = content.replace(FIELD_MARK, ' ').replace(RECORD_MARK, ' ')
        f.write(f'{doc.doc_id}{FIELD_MARK}{content}{RECORD_MARK}\n')

print(f'Written {EXCH_FILE} ({EXCH_FILE.stat().st_size:,} bytes)')

## 5. Build the OKAPI database

`convert_runtime` converts the exchange file into OKAPI's binary bibfile format.

In [ ]:
result = subprocess.run(
    ['convert_runtime', '-c', BSS_PARMPATH, DB_NAME],
    stdin=EXCH_FILE.open('rb'),
    capture_output=True, text=True
)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'convert_runtime failed (exit {result.returncode}). '
                       'Are you on x86? OKAPI binaries are 32-bit x86 only.')

## 6. Build indexes

Two passes per index: `ix1` (inverted file construction) piped into `ixf` (posting file assembly).

In [ ]:
def build_index(index_num: int, doclens: bool = False):
    """Run ix1 | ixf for the given index number."""
    ix1_cmd = ['ix1', '-c', BSS_PARMPATH]
    if doclens:
        ix1_cmd.append('-doclens')
    ix1_cmd += [DB_NAME, str(index_num)]

    ixf_cmd = ['ixf', '-c', BSS_PARMPATH, DB_NAME, str(index_num)]

    ix1_proc = subprocess.Popen(ix1_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    ixf_proc = subprocess.Popen(ixf_cmd, stdin=ix1_proc.stdout,
                                 stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    ix1_proc.stdout.close()
    ixf_out, ixf_err = ixf_proc.communicate()
    ix1_proc.wait()

    if ixf_proc.returncode != 0:
        print('ix1 stderr:', ix1_proc.stderr.read().decode())
        print('ixf stderr:', ixf_err.decode())
        raise RuntimeError(f'Indexing failed for index {index_num}')
    print(f'Index {index_num} built OK')

# Index 0: keyword search — needs -doclens for BM25 length normalisation
build_index(0, doclens=True)

# Index 1: docno lookup
build_index(1, doclens=False)

print('\nBibfiles directory contents:')
for p in sorted(BIB_DIR.iterdir()):
    print(f'  {p.name:40s} {p.stat().st_size:>10,} bytes')

## 7. BM25 query function

Two-pass approach:
1. **Pass 1**: `FIND t=<term>` + `WEIGHT` for each query term → extract IDF weights
2. **Pass 2**: `FIND s=0 w=<w0> s=1 w=<w1> ... op=bm25` → `SHOW format=197`

In [ ]:
def _i1plus(commands):
    """Run a list of BSS commands through i1+ and return stdout lines."""
    r = subprocess.run(
        ['i1+', '-silent', '-flush', '-c', BSS_PARMPATH],
        input='\n'.join(commands) + '\n',
        capture_output=True, text=True
    )
    return r.stdout.splitlines()


def run_bm25_query(query_text: str, top_k: int = 1000):
    """
    Run BM25 retrieval for `query_text`.
    Returns [(docno, score), ...] sorted by score descending.
    """
    terms = [t for t in query_text.lower().split() if t]
    if not terms:
        return []

    # ── Pass 1: get IDF weight for each term ────────────────────────────────
    cmds1 = [f'CHOOSE {DB_NAME}']
    for term in terms:
        cmds1.append(f'FIND t={term}')
        cmds1.append('WEIGHT')

    lines1 = _i1plus(cmds1)

    # Parse alternating FIND / WEIGHT lines
    set_weights = []  # [(set_num, weight), ...]
    i = 0
    while i < len(lines1):
        m = re.match(r'S(\d+)\s+np=\d+', lines1[i])
        if m and i + 1 < len(lines1):
            try:
                w = float(lines1[i + 1].strip())
                set_weights.append((int(m.group(1)), w))
                i += 2
                continue
            except ValueError:
                pass
        i += 1

    if not set_weights:
        return []  # no terms found in index

    # ── Pass 2: BM25 combine and retrieve ───────────────────────────────────
    cmds2 = [f'CHOOSE {DB_NAME}']
    for term in terms:
        cmds2.append(f'FIND t={term}')  # recreate sets (same order → same ids)

    combine = 'FIND ' + ' '.join(f's={s} w={w:.4f}' for s, w in set_weights)
    combine += f' op=bm25 target={top_k}'
    cmds2.append(combine)
    cmds2.append(f'SHOW format=197 n={top_k}')

    lines2 = _i1plus(cmds2)

    # SHOW format=197: "<docno> <score>" per line
    results = []
    for line in lines2:
        parts = line.strip().split()
        if len(parts) == 2:
            try:
                results.append((parts[0], float(parts[1])))
            except ValueError:
                pass

    return sorted(results, key=lambda x: -x[1])


# Quick smoke test
test_results = run_bm25_query('aerodynamic forces compressible flow')
print(f'Query returned {len(test_results)} results')
for docno, score in test_results[:5]:
    print(f'  {docno:15s} {score:.3f}')

## 8. Run all 225 Cranfield queries

In [ ]:
from ir_measures import ScoredDoc

run = []
queries = list(dataset.queries_iter())

for q in queries:
    ranked = run_bm25_query(q.text, top_k=1000)
    for docno, score in ranked:
        run.append(ScoredDoc(query_id=q.query_id, doc_id=docno, score=score))

print(f'Queries: {len(queries)}, total scored docs: {len(run)}')

## 9. Evaluate with NDCG

In [ ]:
import ir_measures
from ir_measures import nDCG

measures = [nDCG@10, nDCG@100, nDCG@1000]
results = ir_measures.calc_aggregate(measures, dataset.qrels_iter(), run)

print('BM25 on Cranfield:')
for measure, value in results.items():
    print(f'  {str(measure):15s} {value:.4f}')